
# 00 — Shared Data Preparation and Integrity Verification

This notebook reconstructs and verifies the frozen data foundation used by all four models. It does **not** silently create a different split. It audits the pinned Kaggle dataset, verifies the committed manifests and reproduces the key integrity checks.


In [ ]:
%pip install -q kagglehub ImageHash

In [ ]:

import os, sys, json, hashlib, random, platform
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import imagehash
import kagglehub
import tensorflow as tf

SEED=42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)
try: tf.config.experimental.enable_op_determinism()
except Exception: pass
print('Python:', sys.version.split()[0])
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:

from pathlib import Path

def locate_project_root():
    """Find the repository root by requiring splits/ and config/ to exist."""
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        Path('/content/brain_tumour_model_comparison'),
        Path('/content/brain-tumor-model-comparison'),
    ])
    content = Path('/content')
    if content.exists():
        candidates.extend([p for p in content.iterdir() if p.is_dir()])

    seen = set()
    for candidate in candidates:
        try:
            candidate = candidate.resolve()
        except Exception:
            continue
        if candidate in seen:
            continue
        seen.add(candidate)
        if (
            (candidate / 'splits' / 'train.csv').is_file()
            and (candidate / 'splits' / 'val.csv').is_file()
            and (candidate / 'splits' / 'test.csv').is_file()
            and (candidate / 'config' / 'class_to_index.json').is_file()
        ):
            return candidate
    raise FileNotFoundError(
        'Could not locate the project root. Clone/upload the repository so that '
        'splits/ and config/ are available, then run the notebook again.'
    )

PROJECT_ROOT = locate_project_root()
SPLITS_DIR = PROJECT_ROOT / 'splits'
CONFIG_DIR = PROJECT_ROOT / 'config'
print('Project root:', PROJECT_ROOT)

EXPECTED_HASHES = {
    'train.csv':'7273ef5fc2cb605c5b03b22a23cda0cb383ac0d9ce5949ac3c95649b5a4270cb',
    'val.csv':'37b24456cfcd1b69df939e36603958eae6a9124b794c32c83a532b9018c6c88f',
    'test.csv':'9afc40a38949eb4f46f8c9591d3b1f51cdd11aa991bfe2cf1d15c386c5537d39',
}
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
for name,expected in EXPECTED_HASHES.items():
    actual=sha256_file(SPLITS_DIR/name)
    print(name, actual, 'MATCH' if actual==expected else 'MISMATCH')
    assert actual==expected


In [ ]:

PINNED_HANDLE='masoudnickparvar/brain-tumor-mri-dataset/versions/1'
DATASET_ROOT=Path(kagglehub.dataset_download(PINNED_HANDLE)).resolve()
print('Dataset root:', DATASET_ROOT)


In [ ]:

train_df=pd.read_csv(SPLITS_DIR/'train.csv')
val_df=pd.read_csv(SPLITS_DIR/'val.csv')
test_df=pd.read_csv(SPLITS_DIR/'test.csv')
classes=['glioma','meningioma','notumor','pituitary']
assert len(train_df)==4353 and len(val_df)==1089 and len(test_df)==1311
assert sorted(train_df.label.unique())==classes
assert sorted(val_df.label.unique())==classes
assert sorted(test_df.label.unique())==classes
print('Frozen counts:', len(train_df), len(val_df), len(test_df))
print(pd.concat({
    'train':train_df.label.value_counts(),
    'val':val_df.label.value_counts(),
    'test':test_df.label.value_counts(),
}, axis=1).reindex(classes))


In [ ]:

# Verify every committed manifest path exists in the pinned dataset.
for split_name,df in [('train',train_df),('val',val_df),('test',test_df)]:
    missing=[fp for fp in df.filepath if not (DATASET_ROOT/fp).is_file()]
    print(split_name, 'missing:', len(missing))
    assert not missing
print('All frozen manifest paths exist.')


In [ ]:

# Verify no cross-split byte-identical hashes remain in the frozen manifests.
train_hash=set(train_df.sha256.astype(str)); val_hash=set(val_df.sha256.astype(str)); test_hash=set(test_df.sha256.astype(str))
print('train∩val:', len(train_hash & val_hash))
print('train∩test:', len(train_hash & test_hash))
print('val∩test:', len(val_hash & test_hash))
assert not (train_hash & val_hash)
assert not (train_hash & test_hash)
assert not (val_hash & test_hash)


In [ ]:

metadata_path=PROJECT_ROOT/'results'/'data_audit'/'dataset_metadata.json'
if metadata_path.exists():
    meta=json.loads(metadata_path.read_text())
    print(json.dumps(meta, indent=2))
else:
    print('dataset_metadata.json not found; frozen manifests are still verified above.')



## Frozen-data rule

The three committed CSV manifests are the group data contract. Model notebooks must reuse them exactly and must never call `train_test_split()` again. The original Kaggle `Testing` directory remains the held-out final test set. Patient-level independence cannot be verified because reliable patient identifiers are unavailable.
